<h3> Project 4 Hacking Food & Nutrition</br>
Group #5: Norman Borlaug </h3>
<h4> Authors: Rishitha Boddu, Omed Akbari, Grace Zhang, Daniel Do, Isha Chhabra </h4>



In [2]:
#imports 

%matplotlib inline

from pathlib import Path
import sys
import subprocess
import importlib.util

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

# install CFEDemands if missing
if importlib.util.find_spec("cfe") is None:
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "CFEDemands"])

import cfe.regression as rgsn

In [5]:
# find all CSV files with "Tanzania" in the name

HOME = Path.home()

csv_files = sorted(HOME.rglob("*.csv"))
tanzania_files = [p for p in csv_files if "tanzania" in p.name.lower()]

if len(tanzania_files) == 0:
    raise FileNotFoundError("No Tanzania CSV files found. Upload the project CSV files first.")

# use the folder with the most Tanzania CSVs
DATA_DIR = max(
    set(p.parent for p in tanzania_files),
    key=lambda folder: sum(1 for p in tanzania_files if p.parent == folder)
)

CSV_FILES = sorted(DATA_DIR.glob("*.csv"))

print("Using data folder:", DATA_DIR)
print("Files found:")
for file in CSV_FILES:
    print(" ", file.name)

Using data folder: /home/jovyan/EEP153_Borlaug/Project 4 - Tanzania/.ipynb_checkpoints
Files found:
  Tanzania - FCT-checkpoint.csv
  Tanzania - Food Expenditures (2019-20)-checkpoint.csv
  Tanzania - Food Expenditures (2020-21)-checkpoint.csv
  Tanzania - Food Prices (2019-20)-checkpoint.csv
  Tanzania - Food Prices (2020-21)-checkpoint.csv
  Tanzania - Household Characteristics-checkpoint.csv
  Tanzania - Region Features-checkpoint.csv


In [7]:
# helper Functions

def find_file(required_words, label):
    """
    finds one CSV file whose filename contains all required words.
    """
    required_words = [word.lower() for word in required_words]

    matches = [
        file for file in CSV_FILES
        if all(word in file.name.lower() for word in required_words)
    ]

    if len(matches) == 0:
        raise FileNotFoundError(f"No file found for {label}")

    matches = sorted(matches, key=lambda p: ("(1)" in p.name, len(p.name)))
    chosen = matches[0]

    print(f"{label}: {chosen.name}")
    return chosen


def clean_columns(df):
    """
    cleans column names and removes unnamed columns.
    """
    df = df.copy()
    df.columns = df.columns.astype(str).str.strip()

    # Remove accidental unnamed columns
    df = df.loc[:, ~df.columns.str.contains("^Unnamed", case=False, regex=True)]

    # Standardize key column names
    rename = {}
    for col in df.columns:
        low = col.lower().strip()
        if low in ["i", "t", "m", "j", "u"]:
            rename[col] = low

    return df.rename(columns=rename)


def clean_id_series(s):
    """
    cleans household, market, time, and food ID columns.
    """
    s = s.astype(str).str.strip()
    s = s.replace({"": np.nan, "nan": np.nan, "NaN": np.nan, "None": np.nan})

    numeric = pd.to_numeric(s, errors="coerce")
    integer_mask = numeric.notna() & np.isclose(numeric, np.round(numeric))

    s.loc[integer_mask] = numeric.loc[integer_mask].round().astype(int).astype(str)

    return s


def clean_id_columns(df, cols):
    """
    applies ID cleaning to selected columns.
    """
    df = df.copy()

    for col in cols:
        if col in df.columns:
            df[col] = clean_id_series(df[col])

    return df


def to_number(s):
    """
    converts messy numeric text into real numbers.
    """
    return pd.to_numeric(
        s.astype(str).str.replace(",", "", regex=False).str.strip(),
        errors="coerce"
    )


def choose_numeric_column(df, id_cols, label):
    """
    chooses the column with the most numeric values.
    """
    candidate_cols = [col for col in df.columns if col not in id_cols]

    if len(candidate_cols) == 0:
        raise ValueError(f"No numeric columns found for {label}")

    numeric_counts = {
        col: to_number(df[col]).notna().sum()
        for col in candidate_cols
    }

    best_col = max(numeric_counts, key=numeric_counts.get)

    print(f"{label} value column:", best_col)
    return best_col